### Assignment 3 - Question 1

##### Deployed Streamlit App
- https://next-word-demo.streamlit.app/

#### Preprocessing and Vocabulary Construction

##### Category I: Natural Language
- Sherlock Holmes (https://www.kaggle.com/datasets/muhammadbilalhaneef/sherlock-holmes-next-word-prediction-corpus/data)




In [2]:
import os
import pickle
import numpy as np
from collections import Counter
from tqdm import tqdm

from datasets import (
    clean_text,
    split_to_sentences,
    build_vocab,
    encode_sentence,
    make_xy,
    word_frequencies,
    top_k_words,
)

# ----------------------------
# CONFIG
# ----------------------------
TEXT_FILE = "./data/sherlock_holmes.txt"
OUTPUT_DIR = "./data/sherlock_data"
CONTEXT_LEN = 5  
MIN_FREQ = 1 

# Verify input file exists
if not os.path.exists(TEXT_FILE):
    raise FileNotFoundError(f"Text file not found: {TEXT_FILE}")

# ----------------------------
# 1. READ TEXT FILE
# ----------------------------
print(f"Reading text from: {TEXT_FILE}")
with open(TEXT_FILE, "r", encoding="utf-8", errors="ignore") as f:
    text = f.read()

print(f"Loaded {len(text):,} characters")

# ----------------------------
# 2. CLEAN AND PREPROCESS TEXT
# ----------------------------
print("Cleaning text...")
cleaned = clean_text(text, keep_periods=True)
print(f"Cleaned text: {len(cleaned):,} characters")

# ----------------------------
# 3. SPLIT INTO SENTENCES
# ----------------------------
print("Splitting into sentences...")
sentences = split_to_sentences(cleaned)
print(f"Found {len(sentences):,} sentences")

# ----------------------------
# 4. BUILD VOCABULARY
# ----------------------------
print("Building vocabulary...")
stoi, itos = build_vocab(sentences, min_freq=MIN_FREQ)

# Get word frequencies for reporting
word_freq = word_frequencies(sentences)
most_frequent, least_frequent = top_k_words(word_freq, k=10)

print(f"Vocabulary size: {len(stoi):,}")
print("Top 10 most frequent words:")
for word, count in most_frequent:
    print(f"   '{word}': {count:,}")
print("Top 10 least frequent words:")
for word, count in least_frequent:
    print(f"   '{word}': {count}")

# ----------------------------
# 5. ENCODE SENTENCES
# ----------------------------
print("Encoding sentences...")
seqs = []
for sent in tqdm(sentences, desc="Encoding"):
    ids = encode_sentence(sent, stoi)
    seqs.append(ids)

total_words = sum(len(s) for s in seqs)
print(f"Encoded {len(seqs):,} sentences ({total_words:,} total word tokens)")

# ----------------------------
# 6. CREATE TRAINING DATA (X, y pairs)
# ----------------------------
print("\n🔨 Creating training data...")
X, y = make_xy(seqs, block_size=CONTEXT_LEN)
print(f"Created {len(X):,} training samples")

# Convert to numpy arrays
X = np.array(X, dtype=np.int32)
y = np.array(y, dtype=np.int32)

# ----------------------------
# 7. SAVE DATA
# ----------------------------
print("Saving dataset...")
os.makedirs(OUTPUT_DIR, exist_ok=True)

X_path = os.path.join(OUTPUT_DIR, "train_X.npy")
y_path = os.path.join(OUTPUT_DIR, "train_y.npy")
vocab_path = os.path.join(OUTPUT_DIR, "vocab.pkl")

np.save(X_path, X)
np.save(y_path, y)

vocab_data = {
    "word2idx": stoi,
    "idx2word": {int(k): v for k, v in itos.items()},
    "context_len": CONTEXT_LEN,
    "word_frequencies": word_freq,
}

with open(vocab_path, "wb") as f:
    pickle.dump(vocab_data, f)

print("\n" + "=" * 60)
print("Dataset ready!")
print("=" * 60)
print(f"Training samples: {X.shape[0]:,}")
print(f"Context length: {CONTEXT_LEN}")
print(f"Features shape: {X.shape}")
print(f"Labels shape: {y.shape}")
print(f"Vocabulary size: {len(stoi):,}")
print("Files saved:")
print(f"   - {X_path}")
print(f"   - {y_path}")
print(f"   - {vocab_path}")
print("=" * 60)


Reading text from: ./data/sherlock_holmes.txt
Loaded 610,871 characters
Cleaning text...
Cleaned text: 544,413 characters
Splitting into sentences...
Found 6,197 sentences
Building vocabulary...
Vocabulary size: 7,902
Top 10 most frequent words:
   'the': 5,632
   'i': 3,036
   'and': 3,020
   'to': 2,747
   'of': 2,660
   'a': 2,644
   'in': 1,766
   'that': 1,752
   'it': 1,736
   'you': 1,504
Top 10 least frequent words:
   '10': 1
   '10th': 1
   '1100': 1
   '12s': 1
   '12th': 1
   '140': 1
   '150': 1
   '16a': 1
   '17': 1
   '1846': 1
Encoding sentences...


Encoding: 100%|██████████| 6197/6197 [00:00<00:00, 308014.38it/s]

Encoded 6,197 sentences (112,237 total word tokens)

🔨 Creating training data...


Created 112,237 training samples
Saving dataset...

Dataset ready!
Training samples: 112,237
Context length: 5
Features shape: (112237, 5)
Labels shape: (112237,)
Vocabulary size: 7,902
Files saved:
   - ./data/sherlock_data/train_X.npy
   - ./data/sherlock_data/train_y.npy
   - ./data/sherlock_data/vocab.pkl


##### Result:
- Vocabulary size: 7902
- Top 10 most frequent words:
   - 'the': 5,632
   - 'i': 3,036
   - 'and': 3,020
   - 'to': 2,747
   - 'of': 2,660
   - 'a': 2,644
   - 'in': 1,766
   - 'that': 1,752
   - 'it': 1,736
   - 'you': 1,504
- Top 10 least frequent words:
   - '10': 1
   - '10th': 1
   - '1100': 1
   - '12s': 1
   - '12th': 1
   - '140': 1
   - '150': 1
   - '16a': 1
   - '17': 1
   - '1846': 1


##### Category II: Structured/Domain Text
- C++ (https://github.com/theAlgorithms/C-Plus-Plus)

In [5]:
import os
import re
import numpy as np
import pickle
from collections import defaultdict, Counter
from tqdm import tqdm

# ----------------------------
# CONFIG
# ----------------------------
DATA_DIR = "./data/cpp_repo"
CONTEXT_LEN = 5 

# Verify directory exists
if not os.path.exists(DATA_DIR):
    raise ValueError(f"Data directory not found: {DATA_DIR}")

# ----------------------------
# 1. COLLECT C++ FILES 
# ----------------------------
cpp_files = []
folder_stats = defaultdict(int)

for root, dirs, files in os.walk(DATA_DIR):
    # Skip hidden directories
    dirs[:] = [d for d in dirs if not d.startswith('.')]

    for f in files:
        if f.endswith((".cpp", ".h", ".hpp")):
            full_path = os.path.join(root, f)
            cpp_files.append(full_path)
            # Track stats per folder
            rel_folder = os.path.relpath(root, DATA_DIR)
            folder_stats[rel_folder] += 1

# print(f"Directory structure: {DATA_DIR}")
# print("Files found per folder:")
# for folder, count in sorted(folder_stats.items()):
#     if folder == '.':
#         print(f"  root: {count} files")
#     else:
#         print(f"  {folder}: {count} files")
# print(f"\n✅ Collected {len(cpp_files)} C++ files total")

# ----------------------------
# 2. READ AND CLEAN CODE
# ----------------------------


def clean_line(line):
    # remove comments and preprocess
    line = re.sub(r'//.*', '', line)
    line = re.sub(r'/\*.*?\*/', '', line)
    line = re.sub(r'\s+', ' ', line.strip())
    return line


all_lines = []
failed_files = []

for file in tqdm(cpp_files, desc="Reading files"):
    try:
        with open(file, 'r', encoding='utf-8', errors='ignore') as f:
            file_lines = 0
            for line in f:
                line = clean_line(line)
                if line:
                    all_lines.append(line)
                    file_lines += 1
    except Exception as e:
        failed_files.append((file, str(e)))
        continue

print(
    f"✅ Processed {len(cpp_files) - len(failed_files)}/{len(cpp_files)} files successfully")
if failed_files:
    print(f"⚠️  Failed to read {len(failed_files)} files")
print(f"📝 Total lines of code: {len(all_lines):,}")

# ----------------------------
# 3. TOKENIZATION
# ----------------------------


def tokenize(line):
    line = re.sub(r'([(){};,+\-*/<>=\[\]])', r' \1 ', line)
    return line.split()


print("\n🔤 Tokenizing...")
tokens = []
for line in tqdm(all_lines, desc="Tokenizing lines"):
    tokens.extend(tokenize(line))

print(f"✅ Total tokens: {len(tokens):,}")

# ----------------------------
# 4. BUILD VOCAB
# ----------------------------
print("\n📚 Building vocabulary...")
vocab = sorted(set(tokens))
word2idx = {w: i for i, w in enumerate(vocab)}
idx2word = {i: w for w, i in word2idx.items()}

print(f"✅ Vocabulary size: {len(vocab):,}")
token_counts = Counter(tokens)
top_tokens = [w for w, _ in token_counts.most_common(10)]
print(f"   Top 10 most frequent tokens: {top_tokens}")
# top 10 least frequent tokens
least_frequent = [w for w, _ in token_counts.most_common()[-10:]]
print(f"   Top 10 least frequent tokens: {least_frequent}")

# ----------------------------
# 5. CREATE TRAINING DATA
# ----------------------------
print("\n🔨 Creating training data...")
X, y = [], []
num_samples = len(tokens) - CONTEXT_LEN

for i in tqdm(range(num_samples), desc="Creating context-target pairs"):
    X.append([word2idx[w] for w in tokens[i:i+CONTEXT_LEN]])
    y.append(word2idx[tokens[i+CONTEXT_LEN]])

X = np.array(X, dtype=np.int32)
y = np.array(y, dtype=np.int32)

print(f"✅ Created {len(X):,} training samples")

# ----------------------------
# 6. SAVE DATA
# ----------------------------
print("\n💾 Saving dataset...")
os.makedirs(DATA_DIR, exist_ok=True)

X_path = os.path.join(DATA_DIR, "train_X.npy")
y_path = os.path.join(DATA_DIR, "train_y.npy")
vocab_path = os.path.join(DATA_DIR, "vocab.pkl")

np.save(X_path, X)
np.save(y_path, y)

with open(vocab_path, "wb") as f:
    pickle.dump({"word2idx": word2idx, "idx2word": idx2word,
                "context_len": CONTEXT_LEN}, f)

print("\n" + "="*60)
print("✅ Dataset ready!")
print("="*60)
print(f"📊 Training samples: {X.shape[0]:,}")
print(f"📏 Context length: {CONTEXT_LEN}")
print(f"📦 Features shape: {X.shape}")
print(f"🎯 Labels shape: {y.shape}")
print(f"📚 Vocabulary size: {len(vocab):,}")
print("\n💾 Files saved:")
print(f"   - {X_path}")
print(f"   - {y_path}")
print(f"   - {vocab_path}")
print("="*60)


Reading files:   0%|          | 0/371 [00:00<?, ?it/s]

Reading files: 100%|██████████| 371/371 [00:00<00:00, 921.23it/s]


✅ Processed 371/371 files successfully
📝 Total lines of code: 51,338

🔤 Tokenizing...


Tokenizing lines: 100%|██████████| 51338/51338 [00:00<00:00, 361051.56it/s]


✅ Total tokens: 370,232

📚 Building vocabulary...
✅ Vocabulary size: 13,831
   Top 10 most frequent tokens: ['*', '(', ';', ')', '<', '=', ',', '>', '{', '}']
   Top 10 least frequent tokens: ['enter\\n"', 'vector:', 'Newton', 'Raphson', 'Currently', '4x', "\\f$f'", '"\\nInitial', 'approximation:', 'elimination']

🔨 Creating training data...


Creating context-target pairs: 100%|██████████| 370227/370227 [00:00<00:00, 430959.41it/s]


✅ Created 370,227 training samples

💾 Saving dataset...

✅ Dataset ready!
📊 Training samples: 370,227
📏 Context length: 5
📦 Features shape: (370227, 5)
🎯 Labels shape: (370227,)
📚 Vocabulary size: 13,831

💾 Files saved:
   - ./data/cpp_repo/train_X.npy
   - ./data/cpp_repo/train_y.npy
   - ./data/cpp_repo/vocab.pkl


##### Result:
Vocabulary size: 13831

Top 10 most frequent tokens:
- '*'
- '('
- ';'
- ')'
- '<'
- '='
- ','
- '>'
- '{'
- '}'

Top 10 least frequent tokens:
- 'enter\\n"'
- 'vector:'
- 'Newton'
- 'Raphson'
- 'Currently'
- '4x'
- "\\f$f'"
- '"\\nInitial'
- 'approximation:'
- 'elimination'


#### Model Design and Training

In [ ]:
from typing import Literal
import torch
from torch import nn


class MLPNextWord(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        block_size: int,
        emb_dim: int = 64,
        hidden_size: int = 1024,
        num_hidden_layers: int = 1,
        activation: Literal["relu", "tanh"] = "relu",
        dropout: float = 0.0,
    ):
        super().__init__()
        self.vocab_size = vocab_size
        self.block_size = block_size
        self.emb = nn.Embedding(vocab_size, emb_dim)

        act = nn.ReLU() if activation == "relu" else nn.Tanh()
        layers = [
            nn.Linear(block_size * emb_dim, hidden_size),
            act,
            nn.Dropout(dropout),
        ]
        for _ in range(max(0, num_hidden_layers - 1)):
            layers.extend([nn.Linear(hidden_size, hidden_size),
                          act, nn.Dropout(dropout)])

        self.mlp = nn.Sequential(*layers)
        self.out = nn.Linear(hidden_size, vocab_size)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, block_size)
        x = self.emb(x)  # (B, block_size, emb_dim)
        x = x.view(x.shape[0], -1)
        x = self.mlp(x)
        logits = self.out(x)
        return logits

In [ ]:
import argparse
import os
import pickle
import json
import numpy as np
import torch
from torch.utils.data import TensorDataset, DataLoader

from models import MLPNextWord
from training import train


def load_cpp_data(data_dir: str):
    """Load preprocessed C++ dataset."""
    X_path = os.path.join(data_dir, "train_X.npy")
    y_path = os.path.join(data_dir, "train_y.npy")
    vocab_path = os.path.join(data_dir, "vocab.pkl")

    if not all(os.path.exists(p) for p in [X_path, y_path, vocab_path]):
        raise FileNotFoundError(
            f"Missing files in {data_dir}. Run prepare_cpp_data.py first."
        )

    X = np.load(X_path)
    y = np.load(y_path)
    with open(vocab_path, "rb") as f:
        vocab_data = pickle.load(f)

    word2idx = vocab_data["word2idx"]
    idx2word = vocab_data["idx2word"]
    context_len = vocab_data.get("context_len", X.shape[1])

    print(f"✅ Loaded dataset from {data_dir}")
    print(f"   Samples: {len(X):,}")
    print(f"   Context length: {context_len}")
    print(f"   Vocabulary size: {len(word2idx):,}")

    return X, y, word2idx, idx2word, context_len


def train_val_split_np(X, y, val_ratio: float = 0.1, seed: int = 42):
    """Split numpy arrays into train/val."""
    rng = np.random.default_rng(seed)
    indices = rng.permutation(len(X))
    n_val = int(len(X) * val_ratio)
    val_idx = indices[:n_val]
    train_idx = indices[n_val:]

    return X[train_idx], y[train_idx], X[val_idx], y[val_idx]


def create_loaders_np(Xtr, ytr, Xval, yval, batch_size: int, device):
    """Create DataLoaders from numpy arrays."""
    dtr = TensorDataset(
        torch.tensor(Xtr, dtype=torch.long, device=device),
        torch.tensor(ytr, dtype=torch.long, device=device),
    )
    dval = TensorDataset(
        torch.tensor(Xval, dtype=torch.long, device=device),
        torch.tensor(yval, dtype=torch.long, device=device),
    )
    return (
        DataLoader(dtr, batch_size=batch_size, shuffle=True, num_workers=0),
        DataLoader(dval, batch_size=batch_size, shuffle=False, num_workers=0),
    )


def main():
    p = argparse.ArgumentParser(description="Train MLP on C++ code dataset")
    p.add_argument(
        "--data_dir",
        type=str,
        default="./data/cpp_repo",
        help="Directory containing train_X.npy, train_y.npy, vocab.pkl",
    )
    p.add_argument("--emb_dim", type=int, default=64,
                   help="Embedding dimension")
    p.add_argument("--hidden", type=int, default=1024,
                   help="Hidden layer size")
    p.add_argument("--layers", type=int, default=4,
                   help="Number of hidden layers")
    p.add_argument("--activation", type=str,
                   default="relu", choices=["relu", "tanh"])
    p.add_argument("--dropout", type=float, default=0.0, help="Dropout rate")
    p.add_argument("--epochs", type=int, default=200, help="Training epochs")
    p.add_argument("--batch_size", type=int, default=2048, help="Batch size")
    p.add_argument("--lr", type=float, default=1e-3, help="Learning rate")
    p.add_argument("--wd", type=float, default=1e-2, help="Weight decay")
    p.add_argument("--val_ratio", type=float, default=0.15,
                   help="Validation split ratio")
    p.add_argument(
        "--ckpt_dir", type=str, default="checkpoints_cpp", help="Checkpoint directory"
    )
    p.add_argument("--seed", type=int, default=42, help="Random seed")
    args = p.parse_args()

    # Set seeds
    torch.manual_seed(args.seed)
    np.random.seed(args.seed)

    # Load data
    X, y, word2idx, idx2word, context_len = load_cpp_data(args.data_dir)

    # Train/val split
    Xtr, ytr, Xval, yval = train_val_split_np(
        X, y, val_ratio=args.val_ratio, seed=args.seed)
    print(f"\n📊 Split: {len(Xtr):,} train, {len(Xval):,} val")

    # Setup device
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"🖥️  Device: {device}")

    # Create loaders
    train_loader, val_loader = create_loaders_np(
        Xtr, ytr, Xval, yval, batch_size=args.batch_size, device=device
    )

    # Create model
    model = MLPNextWord(
        vocab_size=len(word2idx),
        block_size=context_len,
        emb_dim=args.emb_dim,
        hidden_size=args.hidden,
        num_hidden_layers=args.layers,
        activation=args.activation,
        dropout=args.dropout,
    ).to(device)

    # Count parameters
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel()
                           for p in model.parameters() if p.requires_grad)
    print(
        f"\n📐 Model parameters: {total_params:,} total, {trainable_params:,} trainable")

    # Train
    print("\n🚀 Starting training...\n")
    history, best_path = train(
        model,
        train_loader,
        val_loader,
        epochs=args.epochs,
        lr=args.lr,
        weight_decay=args.wd,
        ckpt_dir=args.ckpt_dir,
        print_every=max(1, args.epochs // 20),
    )

    # Save vocab for inference
    vocab_json_path = os.path.join(args.ckpt_dir, "vocab.json")
    with open(vocab_json_path, "w") as f:
        json.dump(
            {
                "word2idx": word2idx,
                "idx2word": {int(k): v for k, v in idx2word.items()},
                "context_len": context_len,
            },
            f,
        )

    print("\n" + "=" * 60)
    print("✅ Training complete!")
    print("=" * 60)
    print(f"📁 Checkpoint: {best_path}")
    print(f"📊 Final train loss: {history['train_loss'][-1]:.4f}")
    print(f"📊 Final val loss: {history['val_loss'][-1]:.4f}")
    print(f"📚 Vocab saved: {vocab_json_path}")
    print("=" * 60)


if __name__ == "__main__":
    main()


##### Model Config: Sherlock Holmes

- Context Length: 5
- Embedding Dimension: 64
- Hidden Size: 1024
- Number of Hidden Layers: 4
- Activation: ReLU
- Output: Softmax over vocabulary


##### Model Config: C++

- Context Length: 5
- Embedding Dimension: 64
- Hidden Size: 512
- Number of Hidden Layers: 4
- Activation: ReLU
- Output: Softmax over vocabulary

##### Training vs Validation Loss Plot

![Training vs Validation Loss Plot](./plots/sherlock_plot.png)

**Validation Loss Pattern:**
The validation loss reveals severe overfitting:
- **Initial Learning (Epochs 0-31)**: Validation loss decreases from ~6.6 to its minimum of 6.2283 at epoch 31, showing initial generalization.
- **Severe Overfitting (Epochs 31-400)**: Validation loss increases dramatically to ~9.3 by epoch 400, while training loss continues decreasing to 3.0. This indicates the model has memorized training examples.

![Training vs Validation Loss Plot](./plots/cpp_plot.png)

**Validation Loss Pattern:**
The validation loss exhibits a different trajectory:
- **Initial Phase (Epochs 0-66)**: Validation loss decreases from ~5.5 to its minimum of 3.2959 at epoch 66, showing good generalization.
- **Overfitting Phase (Epochs 66-100)**: After epoch 66, validation loss plateaus and slightly increases, while training loss continues to decrease. This widening gap indicates overfitting.

##### Embedding Visualization
![Embedding Visualization](./plots/sherlock_embeddings.png)
**Before training:**

Random distribution, no semantic structure

**After training:**

- Semantic clustering observed:
- Human attributes cluster ("witted" + "ladyship")
- Actions cluster ("performed" + "openings")
- Abstract concepts cluster together
- Compact, organized embedding space


![Embedding Visualization](./plots/cpp_embeddings.png)

**Before training:**

Random distribution, no functional structure

**After training:**

- Domain-specific clustering:
- Cryptography terms cluster ("cryptography" + "base64_string")
- Data structures cluster ("parent_of.find" + "list.second")
- STL patterns cluster ("std::find" + related terms)
- Algorithm/math functions cluster


#### Comparative Analysis: Category I vs Category II


##### data
**Sherlock Holmes (Category I - Natural Language):**
- **Dataset**: Literary text from Sherlock Holmes stories
- **Vocabulary Size**: 7,902 unique words
- **Context Length**: 5 words
- **Predictability**: Moderate - language follows grammatical rules but allows creative expression

**C++ Code (Category II - Structured Text):**
- **Dataset**: C++ source code files from algorithms repository
- **Vocabulary Size**: ~13k+ unique tokens (includes keywords, identifiers, operators)
- **Context Length**: 5 tokens
- **Nature**: Structured programming language with strict syntax rules and patterns
- **Predictability**: High - code follows strict patterns, API conventions, and predictable structures

##### Model config
**Sherlock Model:**
- **Hidden Size**: 1,024 neurons
- **Hidden Layers**: 4 layers
- **Total Parameters**: ~12.08M parameters
- **Rationale**: Larger model needed to capture semantic complexity and varied linguistic patterns

**C++ Model:**
- **Hidden Size**: 512 neurons
- **Hidden Layers**: 4 layers
- **Total Parameters**: ~3.84M parameters
- **Rationale**: Smaller model sufficient due to more structured and predictable code patterns


##### Model Performance Comparison

**Sherlock Model Performance:**
- **Training Duration**: 400 epochs
- **Best Validation Loss**: 6.23 at epoch 31
- **Final Validation Loss**: ~9.3 (Overfitting)
- **Final Training Loss**: 3.00
- **Overfitting Severity**: **Severe** - validation loss more than doubled after optimal point
- **Gap at Best**: Train loss ~3.0, Val loss ~6.2 (gap ~3.2)

**C++ Model Performance:**
- **Training Duration**: 100 epochs
- **Best Validation Loss**: 3.30 at epoch 66
- **Final Validation Loss**: ~3.3-3.4
- **Final Training Loss**: 2.13
- **Overfitting Severity**: **Moderate** - validation loss plateaus but doesn't dramatically increase
- **Gap at Best**: Train loss ~2.2, Val loss ~3.3 (gap ~1.1)